<a href="https://colab.research.google.com/github/tony-97/idat-sync/blob/main/colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


In [1]:
%pip install -U git+https://github.com/tony-97/idat-sync moviepy
!playwright install chromium  --with-deps
!apt install weasyprint
!apt-get install xattr

In [ ]:
import asyncio
import mimetypes
import os
from subprocess import getoutput

import ipywidgets as widgets
from IPython.display import display
from google.colab import drive
from moviepy import VideoFileClip
from weasyprint import HTML

from idat_sync.bootstrap import create_auth_service, create_sync_service
from idat_sync.config import load_config
from idat_sync.domain.path_utils import safe_filename

drive.mount("/content/drive", force_remount=True)

config = load_config()
auth_service = create_auth_service(config)

if auth_service.get_credentials():
    print("Already logged in. Run the next cell to select a course and sync.")
else:
    username_input = widgets.Text(description="Username:", placeholder="user@idat.edu.pe")
    password_input = widgets.Password(description="Password:")
    login_button = widgets.Button(description="Login", button_style="primary")
    mfa_box = widgets.VBox([])
    login_output = widgets.Output()

    async def get_mfa_token():
        mfa_future = asyncio.get_running_loop().create_future()
        mfa_input = widgets.Text(description="MFA Code:", placeholder="6-digit code")
        mfa_button = widgets.Button(description="Submit MFA", button_style="success")

        def on_mfa_submit(_):
            if not mfa_future.done():
                mfa_future.set_result(mfa_input.value.strip())
            mfa_box.children = ()

        mfa_button.on_click(on_mfa_submit)
        mfa_box.children = (widgets.HBox([mfa_input, mfa_button]),)
        return await mfa_future

    async def do_login():
        login_button.disabled = True
        with login_output:
            login_output.clear_output()
            print("Logging in...")
            try:
                ok = await auth_service.login(
                    username_input.value.strip(),
                    password_input.value,
                    get_mfa_token,
                )
                if ok and auth_service.get_credentials():
                    print("Login successful! Run the next cell to select a course and sync.")
                else:
                    print("Login failed: Could not obtain credentials.")
                    print(
                        "Please retry logging in later. Because this is running in a Google Colab environment, the datacenter IP might be temporarily blocked."
                    )
            finally:
                mfa_box.children = ()
                login_button.disabled = False

    login_button.on_click(lambda _: asyncio.create_task(do_login()))
    display(widgets.VBox([username_input, password_input, login_button, mfa_box, login_output]))

In [ ]:
def get_share_link(path):
    file_id = getoutput(f'xattr -p "user.drive.id" "{path}"')
    return f"https://drive.google.com/file/d/{file_id}"


def html_to_pdf(path):
    filename = os.path.splitext(path)[0]
    output_path = f"{filename}.pdf"
    HTML(filename=path).write_pdf(output_path)
    print(f"Successfully converted {path} to {output_path}")
    return output_path


def video_to_mp3(path):
    filename = os.path.splitext(path)[0]
    output_audio_path = f"{filename}.mp3"
    video_clip = VideoFileClip(path)
    audio_clip = video_clip.audio
    if audio_clip is None:
        print(f"Skipping: No audio track found in '{path}'.")
        video_clip.close()
        return

    audio_clip.write_audiofile(output_audio_path, logger=None)
    audio_clip.close()
    video_clip.close()
    print(f"Successfully extracted audio to {output_audio_path}")
    return output_audio_path


credentials = auth_service.get_credentials()
if not credentials:
    print("Not logged in. Please complete login in the previous cell first.")
else:
    idat_sync = await create_sync_service(credentials, config)
    courses = await idat_sync.list_courses()
    course_dropdown = widgets.Dropdown(
        options=[(f"[{c.id}] {c.name}", c) for c in courses],
        description="Course:",
        layout=widgets.Layout(width="600px"),
    )
    sync_button = widgets.Button(description="Sync Course", button_style="primary")
    sync_output = widgets.Output()

    async def do_sync():
        sync_button.disabled = True
        try:
            with sync_output:
                sync_output.clear_output()
                course = course_dropdown.value
                course_folder = os.path.join(
                    "drive/MyDrive/sync", safe_filename(course.name)
                )
                print(f"Syncing: {course.name} (id={course.id}) ...")
                async for event in idat_sync.sync_course(course, course_folder):
                    print(event)

                links = []
                recordings = []
                for root, dirs, files in os.walk(course_folder):
                    for file in files:
                        path = os.path.join(root, file)
                        filename, ext = os.path.splitext(path)
                        mime_type, _ = mimetypes.guess_type(path)
                        if ext == ".html":
                            if os.path.exists(f"{filename}.pdf"):
                                continue
                            path = html_to_pdf(path)
                        elif ext == ".mp3":
                            recordings.append(path)
                            continue
                        elif mime_type and mime_type.startswith("video/"):
                            if os.path.exists(f"{filename}.mp3"):
                                continue
                            mp3 = video_to_mp3(path)
                            if mp3:
                                recordings.append(mp3)
                            continue
                        links.append(get_share_link(path))

                recordings = [get_share_link(path) for path in recordings]
                print("=========Recordings=======")
                for recording in recordings:
                    print(recording)
                print("=========Links=======")
                for link in links:
                    print(link)
        finally:
            sync_button.disabled = False

    sync_button.on_click(lambda _: asyncio.create_task(do_sync()))
    display(widgets.VBox([course_dropdown, sync_button, sync_output]))
